In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib3
import openpyxl
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from datetime import timedelta, date
import os
from pathlib import Path
from sklearn.metrics import mean_squared_error
from mssql_python import connect
from nbdevAuto.functions import * 
from io import StringIO
import nbdevAuto.functions
import time 
import lxml

In [ ]:
import duckdb

### Testing .getenv variables

In [2]:
YES_USERNAME = os.getenv("YES_USERNAME")
YES_PASSWORD = os.getenv("YES_PASSWORD")

print("YES_USERNAME loaded:", YES_USERNAME is not None)
print("YES_PASSWORD loaded:", YES_PASSWORD is not None)


YES_USERNAME loaded: True
YES_PASSWORD loaded: True


### Testing Allegro DB connection

I suspect I need to get read access to the server or the database. Who do I need to contact for that???

In [3]:
from mssql_python import connect

# Keep your original string format exactly as you had it
SQL_CONNECTION_STRING = ("Server=HDQv1958;" "Database=allegro;" "Trusted_Connection=yes;" "Encrypt=yes;" "TrustServerCertificate=yes;")

# Crucial fix: Call it explicitly from the imported module
conn = connect(SQL_CONNECTION_STRING)
print("Connected successfully!")


OperationalError: Driver Error: Invalid authorization specification; DDBC Error: [Microsoft][SQL Server]Login failed for user 'BEPC\A105158'.

In [4]:
from mssql_python import connect

SQL_CONNECTION_STRING = ( "Server=HDQv1958;" "Database=allegro;" "Trusted_Connection=yes;" "Encrypt=yes;" "TrustServerCertificate=yes;")
conn = connect(SQL_CONNECTION_STRING)


OperationalError: Driver Error: Invalid authorization specification; DDBC Error: [Microsoft][SQL Server]Login failed for user 'BEPC\A105158'.

In [5]:
import duckdb
import pandas as pd
from mssql_python import connect

# 1. Establish your standard SQL Server Connection using Python
SQL_CONNECTION_STRING = (
    "Server=HDQv1958;"
    "Database=allegro;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

print("Connecting to Microsoft SQL Server via Python...")
sql_conn = connect(SQL_CONNECTION_STRING)

# 2. Extract your target query into a Pandas DataFrame
# (Replace 'your_table' with an actual table in your allegro database)
query = "SELECT TOP 100 * FROM dbo.your_table;"
print("Streaming remote table data into memory...")
df_data = pd.read_sql(query, sql_conn)

# Close the SQL server line as soon as you have the data
sql_conn.close()

# 3. Spin up an offline DuckDB instance 
# (This requires zero internet downloads or remote community extensions)
print("\nBooting local DuckDB engine...")
duck_conn = duckdb.connect(":memory:")

# 4. Query the Pandas DataFrame directly inside DuckDB!
# DuckDB scans your local python variables and treats the dataframe like a table.
duckdb_result = duck_conn.execute("SELECT * FROM df_data WHERE id > 10").df()

print("\n--- Processed Results from DuckDB ---")
print(duckdb_result.head())

duck_conn.close()




Connecting to Microsoft SQL Server via Python...


OperationalError: Driver Error: Invalid authorization specification; DDBC Error: [Microsoft][SQL Server]Login failed for user 'BEPC\A105158'.

In [ ]:

def load_burn():

    query = """
SELECT t.trade, p.marketarea, q.begtime, q.energy, q.quantitystatus

FROM trade t, position p, ngquantity q

WHERE p.bepc_strategy = 'Burn' and t.trade = p.trade and t.tradestatus <> 'Void' and p.position = q.position and q.posstatus = 1
AND q.begtime >= DATEADD(year, -2, GETDATE()) AND q.begtime <= GETDATE()

ORDER BY q.begtime"""

    df = pd.read_sql(query, conn)
    #df["datetime"] = pd.to_datetime(df["datetime"])

    return df[["begtime", "marketarea", "energy"]]

In [ ]:
gas_daily_df = load_burn()
gas_daily_df.head(7)

### Testing YES Energy Connection

In [4]:
def pull_yes_forecast_historical(user, password, start_date, end_date):

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    dfs = []

    while start <= end:

        month_start = start.replace(day=1)
        month_end = month_start + pd.offsets.MonthEnd(1)

        if month_end > end:
            month_end = end

        print(f"Pulling forecast: {month_start.date()} → {month_end.date()}")

        url = (
            "https://services.yesenergy.com/PS/rest/timeseries/multiple.json"
            "?agglevel=hour"
            "&timezone=CPT"
            f"&startdate={month_start.date()}"
            f"&enddate={month_end.date()}"
            "&items="
            "LOAD_FORECAST:10017060648,"
            "NET_LOAD_FORECAST_CURRENT:10017060648,"
            "NG_CAPACITY_OFFLINE:10017060648,"
            "COAL_CAPACITY_OFFLINE:10017060648,"
            "WINDFCST_HOURLY:10004185377,"
            "WINDFCST_HOURLY:10004185378,"
            "WINDFCST_HOURLY:10004185379,"
            "WINDFCST_HOURLY:10004185380,"
            "WINDFCST_HOURLY:10004185381,"
            "WSI_FC15_FEEL:10000355230,"
            "WSI_FC15_FEEL:10000355704,"
            "WSI_FC15_FEEL:10000356081,"
            "WSI_FC15_WIND:10000355230,"
            "WSI_FC15_WIND:10000355704" )

        # retry block in case the api crashes
        for attempt in range(3):
            try:
                resp = requests.get(url, auth=(user, password), verify=False, timeout=120)
                resp.raise_for_status()
                break

            except requests.exceptions.RequestException as e:
                print(f"Retry {attempt+1}/3 failed: {e}")
                time.sleep(5)

                if attempt == 2:
                    raise e

        # processing after successful data pull
        df_chunk = pd.DataFrame(resp.json())
        df_chunk.columns = df_chunk.columns.map(lambda x: str(x).strip())

        dfs.append(df_chunk)

        time.sleep(1)  

        start = month_end + timedelta(days=1)

    return pd.concat(dfs, ignore_index=True)

In [5]:

#call the API
df_forecast_raw = pull_yes_forecast_historical( YES_USERNAME, YES_PASSWORD, start_date="2026-04-01", end_date="2026-06-01")

Pulling forecast: 2026-04-01 → 2026-04-30
Pulling forecast: 2026-05-01 → 2026-05-31
Pulling forecast: 2026-06-01 → 2026-06-01


In [27]:
df_forecast_raw.head(7)

,DATETIME,SPPISO-East (LOAD_FORECAST),SPPISO-East (NET_LOAD_FORECAST_CURRENT),SPPISO-East (NG_CAPACITY_OFFLINE),SPPISO-East (COAL_CAPACITY_OFFLINE),RESERVE ZONE 1 (WINDFCST_HOURLY),RESERVE ZONE 2 (WINDFCST_HOURLY),RESERVE ZONE 3 (WINDFCST_HOURLY),RESERVE ZONE 4 (WINDFCST_HOURLY),RESERVE ZONE 5 (WINDFCST_HOURLY),ND - Bismarck/Municipal (WSI_FC15_FEEL),ND - Fargo/Hector Field (WSI_FC15_FEEL),ND - Williston/Sloulin (WSI_FC15_FEEL),ND - Bismarck/Municipal (WSI_FC15_WIND),ND - Fargo/Hector Field (WSI_FC15_WIND),HOURENDING,MARKETDAY,PEAKTYPE,MONTH,YEAR
0,04/01/2026 01:00:00,29870,13165.23,14352.96,8114.8,1955.22,3943.98,888.02,7533.82,2383.73,19.32,17.37,16.09,11.2,8.7,1,04/01/2026,None,APRIL,2026
1,04/01/2026 02:00:00,29140,13446.82,14352.96,8114.8,1501.2,3638.72,831.6,7372.25,2349.41,17.92,17.34,16.37,12.4,8.1,2,04/01/2026,None,APRIL,2026
2,04/01/2026 03:00:00,28645,13890.99,14352.96,8114.8,1489.96,3173.92,495.43,7325.76,2268.94,18.12,17.34,16.32,11.8,8.1,3,04/01/2026,None,APRIL,2026
3,04/01/2026 04:00:00,28442,14638.86,14185.96,8114.8,1363.78,2794.9,277.28,6931.83,2435.35,17.1,17.12,15.85,13,8.1,4,04/01/2026,None,APRIL,2026
4,04/01/2026 05:00:00,28519,15038.93,14185.96,8114.8,1556.68,2757.74,248.05,6433.37,2484.23,20.76,18.03,17.26,10.6,8.7,5,04/01/2026,None,APRIL,2026
5,04/01/2026 06:00:00,29139,16387.99,13752.45,8488.8,1495.33,2548.91,165.33,5909.02,2632.42,20.22,17.81,17.26,11.2,8.7,6,04/01/2026,None,APRIL,2026
6,04/01/2026 07:00:00,30814,18973.934167,13762.45,8488.8,1216.43,2399.09,150.68,5184.53,2889.3,18.65,18.88,16.78,11.2,8.1,7,04/01/2026,None,APRIL,2026


### YES Energy API


#### Object Search

In [4]:
base_url = "https://services.yesenergy.com/PS/rest/objects"

extension = ["pricenodes", "sppiso"]

full_url = f"{base_url}/{'/'.join(extension)}"
print(full_url)

resp = requests.get(full_url, auth=(YES_USERNAME, YES_PASSWORD), verify=False, timeout=120)
html = resp.content.decode('utf-8')
df = pd.DataFrame(pd.read_html(StringIO(html))[0])

# converting datatypes
column_types = {
    'OBJECTID': 'str',
    'NODENAME': 'str',
    'ZONE': 'str',
    'ZONEID': 'str',
    'NODETYPE': 'str',
    'SUBSTATION': 'str',
    'VOLTAGE': 'str',
    'EQUIPMENT': 'str',
    'FIRST_DART_DATE': 'str',
    'LAST_DART_DATE': 'str',
    'NEAREST_WEATHERSTATION': 'str',
    'WEATHERSTATIONID': 'str',
    'ISO': 'str'
}
df = df.astype(dtype = column_types)

# dates
df['FIRST_DART_DATE'] = pd.to_datetime(df['FIRST_DART_DATE'], errors='coerce')
df['LAST_DART_DATE'] = pd.to_datetime(df['LAST_DART_DATE'], errors='coerce')


# filtering to current records
#last_date = pd.to_datetime(date.today() + timedelta(days=1)).date()
last_date = np.max(df['LAST_DART_DATE'])
df = df[df['LAST_DART_DATE'] == last_date]
print(last_date)

print(f'dimensions are:  {df.shape}')
print(f"Column names:  {' | '.join(df.columns)}")
#print(df.dtypes)
df.head(5)

https://services.yesenergy.com/PS/rest/objects/pricenodes/sppiso
2026-07-16 00:00:00
dimensions are:  (10821, 13)
Column names:  OBJECTID | NODENAME | ZONE | ZONEID | NODETYPE | SUBSTATION | VOLTAGE | EQUIPMENT | FIRST_DART_DATE | LAST_DART_DATE | NEAREST_WEATHERSTATION | WEATHERSTATIONID | ISO


,OBJECTID,NODENAME,ZONE,ZONEID,NODETYPE,SUBSTATION,VOLTAGE,EQUIPMENT,FIRST_DART_DATE,LAST_DART_DATE,NEAREST_WEATHERSTATION,WEATHERSTATIONID,ISO
0,10002510961,AEC,NaN,NaN,INTERFACE,NaN,NaN,NaN,2013-10-01 01:00:00,2026-07-16,IN - Evansville/Regional,10000355676.0,SPPISO
1,10002510962,AECC_CSWS,CSWS,10001836758.0,ZONE,NaN,NaN,NaN,2013-10-01 01:00:00,2026-07-16,AR - Fort Smith/Municipal,10000355773.0,SPPISO
2,10002902516,AECC_ELKINS,CSWS,10001836758.0,GENERATOR,ELKINS,NaN,NaN,2015-12-01 01:00:00,2026-07-16,AR - Fort Smith/Municipal,10000355773.0,SPPISO
6,10002510993,AECC_FITZHUGH,CSWS,10001836758.0,GENERATOR,Fitzhugh CoOp,NaN,NaN,2013-10-01 01:00:00,2026-07-16,AR - Fort Smith/Municipal,10000355773.0,SPPISO
7,10002510994,AECC_FLTCREEK,CSWS,10001836758.0,GENERATOR,Flint Creek,NaN,NaN,2013-10-01 01:00:00,2026-07-16,MO - Joplin/Regional,10000356116.0,SPPISO


In [145]:
base_url = "https://services.yesenergy.com/PS/rest/objects"

extension = ["genunits", "sppiso"]

searches = ["?state=ND", "&status=Operating"]

if searches:
    full_url = f"{base_url}/{'/'.join(extension)}{''.join(searches)}"
else:
    full_url = f"{base_url}/{'/'.join(extension)}"
print(full_url)


https://services.yesenergy.com/PS/rest/objects/genunits/sppiso?state=ND&status=Operating


In [146]:
resp = requests.get(full_url, auth=(YES_USERNAME, YES_PASSWORD), verify=False, timeout=120)
print(resp.status_code)
html = resp.content.decode('utf-8')
df = pd.DataFrame(pd.read_html(StringIO(html))[0])

200


In [147]:
print(df.shape)
df.head()

(62, 21)


,NAMEPLATECAPACITY,ONLINEDATE,PLANTCAPACITY,PLANTCODE,PLANTNAME,PLANTNUMUNITS,PLANTOBJECTID,NODENAME,PNODEOBJECTID,PRIMARYFUEL,...,SECONDARYFUEL,STATE,STATUS,SUMMERCAPACITY,UNITNAME,UNITOBJECTID,UTILITYNAME,WINTERCAPACITY,ZONE,ISO
0,477.0,07/01/1984 00:00:00,1908.0,6469,Antelope Valley,2,10001867134,WAUE.BEPM.AVS1,1.000288e+10,Coal,...,Coal,ND,Operating,450.0,1,10000769220,Basin Electric Power Coop,450.0,WAUE,SPPISO
1,477.0,07/01/1986 00:00:00,1908.0,6469,Antelope Valley,2,10001867134,WAUE.BEPM.AVS2,1.000288e+10,Coal,...,Coal,ND,Operating,450.0,2,10000790810,Basin Electric Power Coop,450.0,WAUE,SPPISO
2,298.8,01/01/2021 00:00:00,597.6,63258,Aurora Wind Project,1,10016237977,WAUE.AWD1.AURAWIND,1.001644e+10,Wind,...,NaN,ND,Operating,298.8,AURWP,10016238066,"Aurora Wind Project, LLC",298.8,WAUE,SPPISO
3,102.4,12/01/2010 00:00:00,204.8,57347,Baldwin Wind Center,1,10001871803,WAUE.BEPM.BALDWIN,1.000288e+10,Wind,...,NaN,ND,Operating,102.4,GE1,10000791687,Baldwin Wind LLC,102.4,WAUE,SPPISO
4,200.0,11/01/2025 00:00:00,400.0,68182,Bowman Wind,1,10018465146,WAUE.TYRE.BOWMANWIND,1.001899e+10,Wind,...,NaN,ND,Operating,200.0,BWMW,10018465722,NaN,200.0,WAUE,SPPISO


#### Timeseries objects

In [22]:
base_url = "https://services.yesenergy.com/PS/rest/timeseries"
#base_url = "https://services.yesenergy.com/PS/rest/timeseries/multiple?agglevel=hour&timezone=CPT&startdate=2025-01-01&enddate=2025-01-31&items=LOAD_FORECAST:10017060648,NET_LOAD_FORECAST_CURRENT:10017060648,NG_CAPACITY_OFFLINE:10017060648,COAL_CAPACITY_OFFLINE:10017060648,WINDFCST_HOURLY:10004185377,WINDFCST_HOURLY:10004185378,WINDFCST_HOURLY:10004185379,WINDFCST_HOURLY:10004185380,WINDFCST_HOURLY:10004185381,WSI_FC15_FEEL:10000355230,WSI_FC15_FEEL:10000355704,WSI_FC15_FEEL:10000356081,WSI_FC15_WIND:10000355230,WSI_FC15_WIND:10000355704"

path = ['WINDFCST_HOURLY', '10004185381']

items = ['?iso=sppiso', '&agglevel=hour']

full_url = f"{base_url}/{'/'.join(path)}{''.join(items)}"
print(full_url)

https://services.yesenergy.com/PS/rest/timeseries/WINDFCST_HOURLY/10004185381?iso=sppiso&agglevel=hour


In [ ]:
resp = requests.get(full_url, auth=(YES_USERNAME, YES_PASSWORD), verify=False, timeout=120)
html = resp.content.decode('utf-8')
print(resp.status_code)
df = pd.DataFrame(pd.read_html(StringIO(html))[0])
df


200


,DATETIME,DAYOFWEEK,OBJECTID,OBJECTNAME,DATATYPE,AGG_LEVEL,MINVALUE,MAXVALUE,AVGVALUE,TIMEZONE,COUNTVALUE,SUMVALUE,STDDEVVALUE,HOURENDING,MARKETDAY,PEAKTYPE,MONTH,YEAR
0,06/16/2026 01:00:00,TUE,10000355704,ND - Fargo/Hector Field,WSI_FC15_WIND,HOUR,12.4,12.4,12.4,CDT,1,12.4,0,1,06/16/2026,OFFPEAK,JUNE,2026
1,06/16/2026 02:00:00,TUE,10000355704,ND - Fargo/Hector Field,WSI_FC15_WIND,HOUR,10.6,10.6,10.6,CDT,1,10.6,0,2,06/16/2026,OFFPEAK,JUNE,2026
2,06/16/2026 03:00:00,TUE,10000355704,ND - Fargo/Hector Field,WSI_FC15_WIND,HOUR,9.3,9.3,9.3,CDT,1,9.3,0,3,06/16/2026,OFFPEAK,JUNE,2026
3,06/16/2026 04:00:00,TUE,10000355704,ND - Fargo/Hector Field,WSI_FC15_WIND,HOUR,8.1,8.1,8.1,CDT,1,8.1,0,4,06/16/2026,OFFPEAK,JUNE,2026
4,06/16/2026 05:00:00,TUE,10000355704,ND - Fargo/Hector Field,WSI_FC15_WIND,HOUR,6.8,6.8,6.8,CDT,1,6.8,0,5,06/16/2026,OFFPEAK,JUNE,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
739,07/16/2026 20:00:00,THU,10000355704,ND - Fargo/Hector Field,WSI_FC15_WIND,HOUR,11.2,11.2,11.2,CDT,1,11.2,0,20,07/16/2026,ONPEAK,JULY,2026
740,07/16/2026 21:00:00,THU,10000355704,ND - Fargo/Hector Field,WSI_FC15_WIND,HOUR,11.8,11.8,11.8,CDT,1,11.8,0,21,07/16/2026,ONPEAK,JULY,2026
741,07/16/2026 22:00:00,THU,10000355704,ND - Fargo/Hector Field,WSI_FC15_WIND,HOUR,11.2,11.2,11.2,CDT,1,11.2,0,22,07/16/2026,ONPEAK,JULY,2026
742,07/16/2026 23:00:00,THU,10000355704,ND - Fargo/Hector Field,WSI_FC15_WIND,HOUR,11.2,11.2,11.2,CDT,1,11.2,0,23,07/16/2026,OFFPEAK,JULY,2026


In [21]:
df.to_csv("output.csv", index=False)

In [160]:
print(df.shape)
print(df['DATATYPE'].unique)
df[df['DATATYPE']== 'WINDFCST_HOURLY']

(434, 5)
<bound method Series.unique of 0         EIA_GAS_STORAGE
1                CEMS_GEN
2            CEMS_GEN_PLT
3          CEMS_HEATINPUT
4      CEMS_HEATINPUT_PLT
              ...        
429       WETBULB_NORM_05
430       WETBULB_NORM_95
431             WIND_NORM
432          WIND_NORM_05
433          WIND_NORM_95
Name: DATATYPE, Length: 434, dtype: str>


,CATEGORY,SUBCATEGORY,DATATYPE,DESCRIPTION,LONG_DESCRIPTION
153,Generation,Forecast,WINDFCST_HOURLY,Gen - Wind Hourly,Gen - Wind Hourly: 7 Day Wind Forecast (ISO On...


In [ ]:

# converting datatypes
column_types = {
    'OBJECTID': 'str',
    'NODENAME': 'str',
    'ZONE': 'str',
    'ZONEID': 'str',
    'NODETYPE': 'str',
    'SUBSTATION': 'str',
    'VOLTAGE': 'str',
    'EQUIPMENT': 'str',
    'FIRST_DART_DATE': 'str',
    'LAST_DART_DATE': 'str',
    'NEAREST_WEATHERSTATION': 'str',
    'WEATHERSTATIONID': 'str',
    'ISO': 'str'
}
df = df.astype(dtype = column_types)

# dates
df['FIRST_DART_DATE'] = pd.to_datetime(df['FIRST_DART_DATE'], errors='coerce')
df['LAST_DART_DATE'] = pd.to_datetime(df['LAST_DART_DATE'], errors='coerce')


# filtering to current records
#last_date = pd.to_datetime(date.today() + timedelta(days=1)).date()
last_date = np.max(df['LAST_DART_DATE'])
df = df[df['LAST_DART_DATE'] == last_date]
print(last_date)

print(f'dimensions are:  {df.shape}')
print(f"Column names:  {' | '.join(df.columns)}")
#print(df.dtypes)
df.head(5)

#### Unit Availability Data

This data is coming from Excel and CSV files

In [6]:
excel_files = [
    # I modified the date specified for the 2026 files
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 06.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 05.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 04.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 03.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 02.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 01.26.xlsx", #commented out since I had it open for testing
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 12.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 11.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 10.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 09.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 08.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 07.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 06.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 05.25 - UPDATE.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 04.25 - UPDATE.xlsx"]

csv_files = [
    r"G:\Trading\Market Operations\Unit availability\2025\dpm_BEPC_GROUPING_2025030100_2025033123 - March.csv", #commented out since I had it open for testing
    r"G:\Trading\Market Operations\Unit availability\2025\transposed_dpm_BEPC_GROUPING_2025020100_2025022823 feb.csv",
    r"G:\Trading\Market Operations\Unit availability\2025\transposed_dpm_BEPC_GROUPING_2025010100_2025013123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024120100_2024123123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024110100_2024113023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024100100_2024103123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024090100_2024093023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024080100_2024083123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024070100_2024073123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024060100_2024063023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024050100_2024053123.csv"]


In [10]:
def pull_unit_availability(excel_files, csv_files):

    # Excel
    excel_dfs = []
    for file in excel_files:
        df = pd.read_excel(file, sheet_name="Gas HEL Transposed")
        df["source_file"] = file
        excel_dfs.append(df)

    excel_combined = pd.concat(excel_dfs, ignore_index=True)

    # CSV
    csv_dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        df["source_file"] = file
        csv_dfs.append(df)

    csv_combined = pd.concat(csv_dfs, ignore_index=True)

    # Combine + defragment
    combined_df = pd.concat([excel_combined, csv_combined], ignore_index=True)
    combined_df = combined_df.copy() 

    return combined_df

In [15]:
csv_files[0:2]

['G:\\Trading\\Market Operations\\Unit availability\\2025\\dpm_BEPC_GROUPING_2025030100_2025033123 - March.csv',
 'G:\\Trading\\Market Operations\\Unit availability\\2025\\transposed_dpm_BEPC_GROUPING_2025020100_2025022823 feb.csv']

In [23]:
df = pull_unit_availability(excel_files[0:2], csv_files[0:2])
#df['DateTime'] = pd.to_datetime(df['DateTime'])
print(df.dtypes)
df.head(7)

DateTime                             object
    CGS1 - High Effective Limit       int64
    DCS1 - High Effective Limit       int64
    GGS1 - High Effective Limit       int64
    GGS2 - High Effective Limit       int64
    LCS1 - High Effective Limit       int64
    LCS2 - High Effective Limit       int64
    LCS3 - High Effective Limit       int64
    LCS4 - High Effective Limit       int64
    LCS5 - High Effective Limit       int64
    LCS6 - High Effective Limit       int64
    PGS1 - High Effective Limit       int64
    PGS2 - High Effective Limit       int64
    PGS3 - High Effective Limit       int64
    PGS11 - High Effective Limit    float64
    PGS12 - High Effective Limit    float64
    PGS13 - High Effective Limit    float64
    PGS14 - High Effective Limit    float64
    PGS15 - High Effective Limit    float64
    PGS16 - High Effective Limit    float64
    PGS17 - High Effective Limit    float64
    PGS18 - High Effective Limit    float64
    PGS19 - High Effective Limit

,DateTime,CGS1 - High Effective Limit,DCS1 - High Effective Limit,GGS1 - High Effective Limit,GGS2 - High Effective Limit,LCS1 - High Effective Limit,LCS2 - High Effective Limit,LCS3 - High Effective Limit,LCS4 - High Effective Limit,LCS5 - High Effective Limit,...,PGS22 - High Effective Limit,PGS31 - High Effective Limit,PGS32 - High Effective Limit,PGS33 - High Effective Limit,PGS34 - High Effective Limit,PGS35 - High Effective Limit,PGS36 - High Effective Limit,PGS4 - High Effective Limit,PGS5 - High Effective Limit,source_file
0,2026-05-01 00:00:00,0,297,58,78,40,45,38,0,0,...,8.9,18.6,0.0,18.6,18.6,18.6,18.6,225.0,225.0,G:\Trading\Market Operations\Unit availability...
1,2026-05-01 01:00:00,0,297,58,78,40,45,42,0,0,...,8.9,18.6,0.0,18.6,18.6,18.6,18.6,225.0,225.0,G:\Trading\Market Operations\Unit availability...
2,2026-05-01 02:00:00,0,297,58,88,40,45,42,0,0,...,8.9,18.6,0.0,18.6,18.6,18.6,18.6,225.0,225.0,G:\Trading\Market Operations\Unit availability...
3,2026-05-01 03:00:00,0,297,61,95,45,45,42,0,0,...,8.9,18.6,0.0,18.6,18.6,18.6,18.6,225.0,225.0,G:\Trading\Market Operations\Unit availability...
4,2026-05-01 04:00:00,0,297,61,95,45,45,42,0,0,...,8.9,18.6,0.0,18.6,18.6,18.6,18.6,225.0,225.0,G:\Trading\Market Operations\Unit availability...
5,2026-05-01 05:00:00,0,297,95,95,45,45,45,0,0,...,8.9,18.6,0.0,18.6,18.6,18.6,18.6,225.0,225.0,G:\Trading\Market Operations\Unit availability...
6,2026-05-01 06:00:00,0,297,95,95,45,45,45,0,0,...,8.9,18.6,0.0,18.6,18.6,18.6,18.6,225.0,225.0,G:\Trading\Market Operations\Unit availability...


In [19]:
print("\n===== AVAILABILITY SOURCE DATA =====")

print(df["DateTime"].min())
print(df["DateTime"].max())

print(df.loc[pd.to_datetime(df["DateTime"], errors="coerce") > "2030-01-01", ["DateTime", "source_file"]].head(100))


===== AVAILABILITY SOURCE DATA =====


TypeError: '<=' not supported between instances of 'Timestamp' and 'str'